# Four Pillars of OOP — Deep Dive

## 0. What is OOP?

**Object-Oriented Programming (OOP)** is a programming paradigm that organizes code around **objects** — bundles of **data (attributes)** and **behavior (methods)** — rather than around functions and logic alone.

### The Four Pillars
1. **Encapsulation** — bundling data + behavior, restricting direct access
2. **Abstraction** — hiding complexity, exposing only essentials
3. **Inheritance** — reusing and extending behavior from existing classes
4. **Polymorphism** — same interface, different underlying implementations

### E.A.I.P — Encapsulate the mess, Abstract away the detail, Inherit what's shared, Polymorphism lets one interface wear many faces.

## 1. Encapsulation

### Definition
**Encapsulation** is the bundling of data (attributes) and the methods that operate on that data into a single unit (a class), while **restricting direct access** to some of an object's internal state to protect it from unintended interference.

### Why It Matters
- Protects internal state from invalid/inconsistent modification
- Hides implementation details — the internal representation can change without breaking external code
- Enforces controlled access via **getters/setters** or **properties**
- Reduces coupling between components


### Access Control in Python

Python doesn't have true `private`/`protected`/`public` keywords like Java or C++ — it relies on **naming conventions**:

| Convention | Syntax | Meaning |
|---|---|---|
| Public | `self.name` | Freely accessible from anywhere |
| Protected (convention) | `self._name` | "Internal use" — a hint to other devs, not enforced |
| Private (name-mangled) | `self.__name` | Python renames it internally to `_ClassName__name` to discourage (not prevent) external access |

In [6]:

class BankAcc :
    def __init__(self, name, balance):
        self.name = name 
        self.__balance = balance
        self._acc_type = 'savings'
    
    def deposit(self, ammount):
        if ammount <= 0:
            raise ValueError('deposit amt must be more than 0')
        self.__balance += ammount
    
    def withdraw(self, ammount):
        if ammount > self.__balance:
            raise ValueError("insufficent fund ")
        self.__balance -= ammount
    @property
    def get_balance(self):
        return self.__balance


In [9]:

ob = BankAcc('mk',1000)

print(ob.get_balance)
ob.deposit(500)
print(ob.get_balance)

1000
1500


### The Pythonic Way — `@property`

Rather than explicit `get_x()`/`set_x()` methods (common in Java), Python uses **properties** to expose controlled access while keeping attribute-style syntax.

In [10]:
class Temperature:
    def __init__(self,celsius):
        self._celsius = celsius
    
    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self,value):
        if value < -275:
            raise ValueError('temp can not be bellow absolute 0')
        self._celsius = value

    @property
    def tempNew(self):
        return self._celsius * 10


        

In [16]:
ab = Temperature(50)

print(ab.celsius)

ab.celsius =100
print(ab.celsius)

print(ab.tempNew)

50
100
1000


## 2. Abstraction

### Definition
**Abstraction** means hiding complex implementation details and exposing only the essential features/interface needed to use an object — the "what" without the "how."

### Abstraction vs Encapsulation (Commonly Confused)

| Aspect | Encapsulation | Abstraction |
|---|---|---|
| Focus | **How** data is protected/bundled | **What** is exposed to the user |
| Mechanism | Access modifiers, `@property` | Abstract classes, interfaces |
| Goal | Data hiding & integrity | Complexity hiding & design simplification |
| Level | Implementation-level | Design-level |

Encapsulation = wrapping data with methods and restricting access.
Abstraction = showing only relevant details and hiding the rest of the design/logic.

### Achieving Abstraction in Python — `abc` module

Python provides the `abc` (Abstract Base Class) module to define abstract classes and methods that **must** be implemented by subclasses.

In [ ]:

from abc import ABC, abstractmethod


class shape(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass

    @property
    def description(self):
        return f"the shape has the area of {self.area()} and the parameter is {self.perimeter()}"



class square(shape):
    def __init__(self,side):
        self.side= side

    def area(self):
        return self.side * self.side

    def perimeter(self):
        return 4* (self.side)



In [24]:
sq = square(5)

sq.description()

'the shape has the area of 25 and the parameter is 20'

### "Duck Typing" — Python's Informal Abstraction

Python often relies on **duck typing** instead of strict interfaces: *"If it walks like a duck and quacks like a duck, it's a duck."* If an object has the right methods, Python doesn't care about its exact type.

In [25]:
class Duck:
    def sound(self):
        return "Quack!"

class Dog:
    def sound(self):
        return "Woof!"

def make_it_speak(animal):
    print(animal.sound())   # works for ANY object with a .sound() method

make_it_speak(Duck())
make_it_speak(Dog())

Quack!
Woof!


here above its just checking in the make_it_speak function that a variable animal which can be any dog or duck have the fucntion sound in them or not , which is animal.sound()

### Abstraction Summary Table

| Aspect | Detail |
|---|---|
| Purpose | Hide complexity, expose only necessary interface |
| Python mechanism | `abc.ABC`, `@abstractmethod`, duck typing |
| Key benefit | Simplifies usage, enforces a contract for subclasses |
| Real-world example | `list.append()` — you don't know/care about internal resizing logic |

## 3. Inheritance

### Definition
**Inheritance** allows a class (child/subclass) to acquire attributes and methods from another class (parent/superclass), promoting **code reuse** and establishing an "is-a" relationship.

A Dog is an Animal. It inherits general animal behaviors (eating, sleeping) while adding/overriding its own specific behavior (barking).

In [ ]:
class Aniaml:
    def __init__(self,name):
        self.name = name 

    def eat(self):
        return f"{self.name} is eating "

    def make_sound(self):
        return "some sound"

class dog(Aniaml):          # Dog inherits from Animal
    def make_sound(self):       # method overriding
        return "bark brak"




In [ ]:
ab = dog("bao")

print(ab.eat())     # inherited method -> "Rex is eating."

ab.make_sound()     # overridden method -> "Woof!"

bao is eating 


'bark brak'

### abstract vs ducktyping 

"The core difference is how and when the interface contract is enforced.

Abstract methods use Python's abc module to enforce an explicit, formal contract via inheritance. Verification happens at instantiation time—if a subclass doesn't implement an abstract method, Python prevents object creation with a TypeError.

Duck typing, on the other hand, is Python's dynamic philosophy of 'if it walks like a duck and quacks like a duck, it's a duck.' There is no inheritance or upfront check; Python only cares that the object provides the required method at runtime execution, raising an AttributeError only if the method is missing when called."

### `super()` — Calling the Parent Class

In [28]:
class Animal:
    def __init__(self, name, species):
        self.name = name
        self.species = species

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, species="Dog")  # reuse parent's init logic
        self.breed = breed

d = Dog("Rex", "Labrador")
print(d.name, d.species, d.breed)   # Rex Dog Labrador

Rex Dog Labrador


Why use super() instead of Animal.__init__(self, ...)?
super() correctly follows the Method Resolution Order (MRO), especially important in multiple inheritance — hardcoding the parent class name bypasses MRO and can cause bugs in complex hierarchies.

### Types of Inheritance

In [2]:
# 1. Single Inheritance
class A: pass
class B(A): pass

# 2. Multilevel Inheritance
class A: pass
class B(A): pass
class C(B): pass          # C -> B -> A

# 3. Multiple Inheritance
class A: pass
class B: pass
class C(A, B): pass       # C inherits from both A and B

# 4. Hierarchical Inheritance
class A: pass
class B(A): pass
class C(A): pass          # B and C both inherit from A

# 5. Hybrid Inheritance -- a combination of the above patterns

### Method Resolution Order (MRO) — Multiple Inheritance

Python uses the **C3 Linearization algorithm** to determine the order in which base classes are searched when calling a method.

In [3]:
class A:
    def greet(self):
        return "Hello from A"

class B(A):
    def greet(self):
        return "Hello from B"

class C(A):
    def greet(self):
        return "Hello from C"

class D(B, C):
    pass

d = D()
print(d.greet())          # "Hello from B" (B comes before C in MRO)
print(D.__mro__)
# (<class 'D'>, <class 'B'>, <class 'C'>, <class 'A'>, <class 'object'>)

Hello from B
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


### Overriding vs Overloading

| Concept                                                                            | Supported in Python?  | Explanation                                                                                                                                                |
| ---------------------------------------------------------------------------------- | --------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Method Overriding                                                                  | Yes                   | Subclass redefines a method inherited from parent                                                                                                          |
| Method Overloading (classic, multiple same-name methods with different signatures) | **No** (not natively) | Python only keeps the **last** defined method with a given name; achieved instead via default arguments, `*args`/`**kwargs`, or `functools.singledispatch` |

In [4]:
# Simulating "overloading" using default args / *args
class Calculator:
    def add(self, a, b=0, c=0):
        return a + b + c

calc = Calculator()
print(calc.add(5))         # 5
print(calc.add(5, 10))      # 15
print(calc.add(5, 10, 15))   # 30

5
15
30


### Composition vs Inheritance
"Favor Composition Over Inheritance"
A well-known OOP design principle. Inheritance creates a tight ("is-a") coupling that can become fragile in deep hierarchies. Composition ("has-a" relationship — building objects using other objects) is often more flexible.

In [5]:
# Inheritance ("is-a")
class Engine:
    def start(self):
        return "Engine starting..."

class Car(Engine):    # awkward: a Car "is-an" Engine? Not really.
    pass

# Composition ("has-a") -- generally preferred here
class Car:
    def __init__(self):
        self.engine = Engine()   # Car HAS an Engine

    def start(self):
        return self.engine.start()

### Inheritance Summary Table

| Aspect | Detail |
|---|---|
| Purpose | Code reuse, establishing "is-a" relationships |
| Python mechanism | `class Child(Parent):`, `super()` |
| Key concepts | MRO, method overriding, multiple inheritance |
| Design tip | Prefer composition when relationship isn't a true "is-a" |

## 4. Polymorphism

### Definition
**Polymorphism** ("many forms") means the same interface (method/function name) can behave differently depending on the object it's called on.


Pressing the "play" button behaves differently for a music app, a video app, or a game console — same action, different underlying behavior depending on context.

### 4.1 Method Overriding (Runtime Polymorphism)

Already seen in inheritance — subclasses provide their own version of a parent's method.


In [ ]:
class Shape:
    def area(self):
        raise NotImplementedError

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return 3.14159 * self.radius ** 2

class Square(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2


shapes = [Circle(5), Square(4)]
for shape in shapes:
    print(shape.area())   # calls the CORRECT area() for each object automatically
# 78.53975
# 16

### 4.2 Duck Typing (Python's Natural Polymorphism)

Since Python doesn't enforce strict types, **any object with the right method** can be used interchangeably — no shared base class required.

In [6]:

class Bird:
    def move(self):
        return "Flies through the sky"

class Fish:
    def move(self):
        return "Swims in water"

class Robot:
    def move(self):
        return "Rolls on wheels"

for entity in [Bird(), Fish(), Robot()]:
    print(entity.move())   # works despite zero shared inheritance

Flies through the sky
Swims in water
Rolls on wheels


### 4.3 Operator Overloading (via Dunder/Magic Methods)

Python lets you redefine how built-in operators behave for custom objects using **dunder methods** (double-underscore / "magic" methods).

In [7]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __add__(self, other):          # overloads the + operator
        return Vector(self.x + other.x, self.y + other.y)

    def __eq__(self, other):            # overloads ==
        return self.x == other.x and self.y == other.y

    def __repr__(self):                  # controls how print() displays it
        return f"Vector({self.x}, {self.y})"


v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)        # Vector(4, 6) -- calls __add__ automatically
print(v1 == Vector(1, 2))   # True -- calls __eq__


Vector(4, 6)
True


### Common Dunder Methods for Polymorphism

| Method | Operator/Behavior |
|---|---|
| `__add__` | `+` |
| `__sub__` | `-` |
| `__mul__` | `*` |
| `__eq__` | `==` |
| `__lt__` / `__gt__` | `<` / `>` |
| `__len__` | `len(obj)` |
| `__str__` | `str(obj)` / `print(obj)` |
| `__repr__` | `repr(obj)` — official/debug string |
| `__getitem__` | `obj[index]` |
| `__iter__` | makes an object iterable (`for x in obj`) |
| `__call__` | lets an object be called like a function: `obj()` |


### 4.4 Polymorphism with Built-in Functions

In [8]:
print(len("hello"))     # 5   -- __len__ on str
print(len([1,2,3]))       # 3   -- __len__ on list
print(len({"a":1,"b":2}))  # 2   -- __len__ on dict

# The SAME function `len()` works differently depending on the object's type

5
3
2


This is a great real-world example: len() is polymorphic because each type implements its own __len__, and len() simply calls whatever version matches the object passed in.

### Polymorphism Summary Table

| Aspect | Detail |
|---|---|
| Purpose | Same interface, different behavior per object |
| Python mechanisms | Method overriding, duck typing, operator overloading (dunder methods) |
| Key benefit | Write generic code that works across many types without type-checking |
| Native Python example | `len()`, `+`, `str()` all behave polymorphically |

## 5. How the Four Pillars Work Together

In [17]:
from abc import ABC, abstractmethod

class Employe(ABC):
    def __init__(self,name,salary):
        self._name = name
        self.__salary = salary
    @property
    def salary(self):
        return self.__salary
    
    @salary.setter
    def salary(self, value):
        if value <=0:
            raise ValueError('salary cant be zero')
        self.__salary=value
    
    @abstractmethod
    def cal_bonus(self):
        pass

    
    def __str__(self):
        return f"{self._name} is the employe and his salalry is {self.salary} and the bonus is {self.cal_bonus()}"



In [19]:
class manager(Employe):
    def cal_bonus(self):
        return self.salary * 0.20

class engineer(Employe):
    def cal_bonus(self):
        return self.salary * 0.25


gbb = [manager('ram',10000), engineer('krish',20000)]

for i in gbb:
    print(i)

ram is the employe and his salalry is 10000 and the bonus is 2000.0
krish is the employe and his salalry is 20000 and the bonus is 5000.0


## 7. Quick Revision Summary

| Pillar | One-Line Definition | Python Mechanism |
|---|---|---|
| **Encapsulation** | Bundle data + behavior, restrict direct access | `_protected`, `__private`, `@property` |
| **Abstraction** | Hide complexity, expose only essentials | `abc.ABC`, `@abstractmethod`, duck typing |
| **Inheritance** | Reuse/extend behavior from another class | `class Child(Parent):`, `super()` |
| **Polymorphism** | Same interface, different behavior per type | Method overriding, dunder methods, duck typing |

**Golden Rules:**
1. Encapsulation ≠ true privacy in Python — it's convention + `@property`
2. Abstraction defines *what*; Inheritance/Polymorphism handle *how* and *which version*
3. `super()` respects MRO — always prefer it over hardcoded parent calls
4. Python favors duck typing — polymorphism doesn't require shared inheritance
5. Favor composition over inheritance when in doubt